<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "450"
iso_country = "Madagascar"
year = 2023
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 16.040*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Madagascar': 

Number of trade records present in dataset: 18972
Number of partner countries present in dataset: 110
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$4072.4m
Total weight within the dataset: 7235.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,450,Madagascar,156,China,Import,Sea,950691,950691 | Athletics and gymnas,9506,"9506 | Gymnastics, athletic",95,"95 | Toys, games and spor",3843,Gymnasium or athletics articles and equipment,Other Manufacturing,42,OMF - Manufactures n.e.c.,752859.845,676995.639,752859.845,kg,440932.57,445865.57,86804.740128,11045.0,6863.042695,595744.637630,['Container'],['Container'],2589.574054,2584.381505,2584.913031,2584.913031,2587.264719,2587.264719,2713.035356,4863.902068,2956.817837,2783.418769,2791.681095,2854.487333,2654.629169,4739.804213,3133.249227,2883.901202,2813.776931,2947.078681,2603.405237,5221.111169,3902.853114,3417.772305,3155.690188,3074.435050,2553.125023,5531.171000,4542.858069,4214.575404,3373.200832,3298.613367,2503.793265,5652.502769,5445.803700,5522.020256,3620.422692,3466.574907,2453.473573,5533.265123,5231.541118,5693.506774,4346.431075,3531.572660,2406.072876,4883.863657,5131.015919,5361.174339,3908.643465,3573.443998,2357.739528,4489.378207,4547.795289,5412.122120,4526.006315,3598.749394,3843
1,1,450,Madagascar,156,China,Import,Sea,960621,960621 | Buttons; of plastics,9606,"9606 | Buttons, press-faste",96,96 | Miscellaneous manufa,38923,"Press-fasteners, snap-fasteners, press- studs ...",Other Manufacturing,42,OMF - Manufactures n.e.c.,255372.022,252754.179,255372.022,kg,17309646.00,16628.23,17084.388272,11045.0,6863.042695,117250.886127,['Container'],['Container'],509.664432,508.642466,508.747078,508.747078,509.209923,509.209923,533.963345,957.284030,581.943151,547.815786,549.441928,561.803075,522.468190,932.859834,616.667319,567.592136,553.790698,580.026349,512.386603,1027.587783,768.136140,672.665444,621.084014,605.091865,502.490753,1088.611899,894.098076,829.487451,663.893154,649.213297,492.781572,1112.491690,1071.810419,1086.810904,712.549878,682.270480,482.877952,1089.024051,1029.640542,1120.561852,855.438493,695.062948,473.548831,961.212750,1009.855776,1055.154176,769.275762,703.303813,464.036152,883.572490,895.069790,1065.181412,890.781415,708.284269,38923


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $848.7m (20.8%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               266863019.1  603135369.2      6.6      8.3
15 | Animal, vegetable or  198879963.3  214080344.3      4.9      3.0
17 | Sugars and sugar con   91134796.8  194003828.1      2.2      2.7
11 | Products of the mill   76514047.7  175269652.5      1.9      2.4
23 | Food industries, res   50174869.9   41369851.7      1.2      0.6
03 | Fish and crustaceans   37531153.1   18019125.9      0.9      0.2
21 | Miscellaneous edible   29139694.1    8362825.4      0.7      0.1
19 | Preparations of cere   23726344.6    9164335.3      0.6      0.1
04 | Dairy produce; birds   22482835.0    8670870.2      0.6      0.1
22 | Beverages, spirits a   19384997.0   28164626.5      0.5      0.4
20 | Preparations of vege   10467395.4    6184029.4      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $691.2m (17.0%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       504381956.3  744978440.5     12.4     10.3
Food & Beverages  149429235.5  101408876.8      3.7      1.4
Fishing            37390187.3   17990847.3      0.9      0.2



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $653.8m (16.1%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     223782469.9  236476524.7      5.5      3.3
VOL - Vegetable oils and fats  196805505.2  212502640.2      4.8      2.9
PCR - Processed rice           190698569.5  331243415.3      4.7      4.6
MIL - Dairy products            23071309.7    8773162.4      0.6      0.1
V_F - Vegetables, fruit, nuts    6393763.7    8156245.5      0.2      0.1
OCR - Crops n.e.c.               5798864.9    1426361.0      0.1      0.0
SGR - Sugar                      2416973.4    5811910.6      0.1      0.1
GRO - Cereal grains n.e.c.       2074676.9    5221389.8      0.1      0.1
OMT - Meat products n.e.c.       1221498.6     305472.6      0.0      0.0
PDR - Paddy rice                  720231.0    1251041.3      0.0      0.0
FSH - Fishing                     497415.0    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 20.8% 
GTAP 16.1% 
EORA 17.0%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46391/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,450,Madagascar,156,China,Import,Sea,950691,950691 | Athletics and gymnas,9506,"9506 | Gymnastics, athletic",95,"95 | Toys, games and spor",3843,Gymnasium or athletics articles and equipment,Other Manufacturing,42,OMF - Manufactures n.e.c.,752859.845,676995.639,752859.845,kg,440932.57,445865.57,86804.740128,11045.0,6863.042695,595744.637630,['Container'],['Container'],2589.574054,2584.381505,2584.913031,2584.913031,2587.264719,2587.264719,2713.035356,4863.902068,2956.817837,2783.418769,2791.681095,2854.487333,2654.629169,4739.804213,3133.249227,2883.901202,2813.776931,2947.078681,2603.405237,5221.111169,3902.853114,3417.772305,3155.690188,3074.435050,2553.125023,5531.171000,4542.858069,4214.575404,3373.200832,3298.613367,2503.793265,5652.502769,5445.803700,5522.020256,3620.422692,3466.574907,2453.473573,5533.265123,5231.541118,5693.506774,4346.431075,3531.572660,2406.072876,4883.863657,5131.015919,5361.174339,3908.643465,3573.443998,2357.739528,4489.378207,4547.795289,5412.122120,4526.006315,3598.749394,3843,NaN
1,1,450,Madagascar,156,China,Import,Sea,960621,960621 | Buttons; of plastics,9606,"9606 | Buttons, press-faste",96,96 | Miscellaneous manufa,38923,"Press-fasteners, snap-fasteners, press- studs ...",Other Manufacturing,42,OMF - Manufactures n.e.c.,255372.022,252754.179,255372.022,kg,17309646.00,16628.23,17084.388272,11045.0,6863.042695,117250.886127,['Container'],['Container'],509.664432,508.642466,508.747078,508.747078,509.209923,509.209923,533.963345,957.284030,581.943151,547.815786,549.441928,561.803075,522.468190,932.859834,616.667319,567.592136,553.790698,580.026349,512.386603,1027.587783,768.136140,672.665444,621.084014,605.091865,502.490753,1088.611899,894.098076,829.487451,663.893154,649.213297,492.781572,1112.491690,1071.810419,1086.810904,712.549878,682.270480,482.877952,1089.024051,1029.640542,1120.561852,855.438493,695.062948,473.548831,961.212750,1009.855776,1055.154176,769.275762,703.303813,464.036152,883.572490,895.069790,1065.181412,890.781415,708.284269,38923,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $594.0m of $4072.4m (14.6%)
Volume: 1078.1 of 7235.2 million kilograms (14.9%)
Value-to-Weight: 0.55 from 0.56 US$ per kg (+-2.1%)
Average distance to market (volume-weighted): 5620.8 from 3933.6 nm (+42.9%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    370  313842230.1  517701565.3      7.7      7.2
1      Sugar and Honey     89   91160386.7  194011164.8      2.2      2.7
2        Fats and Oils    104   82326529.8   76889047.4      2.0      1.1
3  Unprocessed Cereals     32   75444218.5  270640912.6      1.9      3.7
4                Dairy     68   22374209.5    8632445.9      0.5      0.1
5           Vegetables    122    6209003.6    8060490.6      0.2      0.1
6               Fruits     97    2264689.1    1803629.8      0.1      0.0
7        Meat and Prep      7     419450.2     344996.1      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,2.216674e+07,3.973891e+07,2.415832e+07,2.274158e+07,2.280909e+07,2.332236e+07,2.045725e+07,4.618259e+07,4.449343e+07,4.511630e+07,2.958010e+07,2.832303e+07,19264006.7,3.667923e+07,3.715696e+07,4.421760e+07,3.697849e+07,2.940289e+07
1,Fats and Oils,1.964100e+08,3.214360e+08,2.097472e+08,1.971141e+08,1.976538e+08,2.041608e+08,1.842521e+08,3.616714e+08,3.543669e+08,3.668759e+08,2.513025e+08,2.391744e+08,175765727.1,3.011934e+08,3.048093e+08,3.534135e+08,2.970481e+08,2.465956e+08
2,Fruits,2.528050e+06,4.513499e+06,2.752051e+06,2.591305e+06,2.599088e+06,2.658372e+06,2.334804e+06,5.246764e+06,5.052956e+06,5.128501e+06,3.367964e+06,3.223208e+06,2199985.9,4.173085e+06,4.228286e+06,5.024054e+06,4.201394e+06,3.346104e+06
3,Meat and Prep,2.703584e+05,4.846958e+05,2.946518e+05,2.773723e+05,2.781956e+05,2.844544e+05,2.495071e+05,5.632812e+05,5.426833e+05,5.502784e+05,3.607811e+05,3.454499e+05,234952.6,4.473739e+05,4.531953e+05,5.393269e+05,4.510240e+05,3.586213e+05
4,Processed Cereals,9.036391e+08,1.603716e+09,9.819928e+08,9.249297e+08,9.276983e+08,9.494631e+08,8.354856e+08,1.867241e+09,1.798170e+09,1.826907e+09,1.204390e+09,1.148288e+09,787918267.5,1.487216e+09,1.507778e+09,1.786851e+09,1.495799e+09,1.191043e+09
5,Sugar and Honey,2.341285e+08,3.639121e+08,2.454303e+08,2.328137e+08,2.335633e+08,2.419250e+08,2.213480e+08,4.351867e+08,4.168367e+08,4.347051e+08,3.081275e+08,2.774415e+08,212427961.1,3.600467e+08,3.689091e+08,4.131728e+08,3.509266e+08,2.842420e+08
6,Unprocessed Cereals,1.083489e+09,1.919043e+09,1.176760e+09,1.108502e+09,1.111820e+09,1.138129e+09,1.002137e+09,2.235343e+09,2.152520e+09,2.187720e+09,1.443973e+09,1.375320e+09,945355639.0,1.781397e+09,1.806331e+09,2.138808e+09,1.790858e+09,1.426212e+09
7,Vegetables,2.890319e+07,5.142129e+07,3.143128e+07,2.959994e+07,2.968892e+07,3.037791e+07,2.671196e+07,5.983232e+07,5.761092e+07,5.850917e+07,3.850413e+07,3.677853e+07,25181341.7,4.761456e+07,4.826458e+07,5.726426e+07,4.790773e+07,3.816316e+07


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.79,0.09,0.03,0.03,0.05,1.15,1.07,1.10,0.41,0.35,0.78,0.80,1.12,0.79,0.45
1,Fats and Oils,1.52,0.16,0.01,0.02,0.09,2.16,2.07,2.22,0.81,0.67,1.52,1.57,2.16,1.47,0.86
2,Fruits,0.88,0.10,0.03,0.03,0.06,1.29,1.20,1.23,0.46,0.39,0.87,0.90,1.25,0.88,0.51
3,Meat and Prep,0.51,0.06,0.02,0.02,0.03,0.75,0.70,0.72,0.27,0.23,0.51,0.52,0.73,0.52,0.29
4,Processed Cereals,2.23,0.25,0.07,0.08,0.15,3.29,3.07,3.16,1.18,1.00,2.23,2.29,3.18,2.26,1.28
5,Sugar and Honey,1.42,0.12,-0.01,-0.01,0.09,2.35,2.14,2.34,0.95,0.62,1.62,1.72,2.20,1.52,0.79
6,Unprocessed Cereals,11.08,1.24,0.33,0.38,0.72,16.35,15.25,15.71,5.86,4.95,11.08,11.41,15.82,11.21,6.37
7,Vegetables,3.63,0.41,0.11,0.13,0.24,5.33,4.98,5.12,1.90,1.62,3.61,3.72,5.17,3.66,2.09
